In [2]:
from pathlib import Path
import duckdb
from IPython.display import display, HTML

ROOT = Path().resolve().parent
db = duckdb.connect(str(ROOT / 'data' / 'mathlib.db'), read_only=True)

## Browse by subarea

Change `SUBAREA` to any of: `Matrix`, `AffineSpace`, `TensorProduct`, `QuadraticForm`,
`CliffordAlgebra`, `Dimension`, `Basis`, `Multilinear`, `Dual`, `BilinearForm`, `Span`, etc.

Set `ONLY_WITH_DOCSTRING = False` to also show declarations that have no docstring
(name + type signature only).

In [3]:
SUBAREA             = 'Matrix'
ONLY_WITH_DOCSTRING = True   # False → show everything, greyed-out when no docstring
LIMIT               = 200    # max rows to render

rows = db.execute("""
    SELECT
        d.name,
        d.kind,
        d.module,
        d.type_sig,
        ds.docstring
    FROM declarations d
    LEFT JOIN docstrings ds ON d.name = ds.name
    WHERE d.module_parts[1] = 'Mathlib'
      AND d.module_parts[2] = 'LinearAlgebra'
      AND d.module_parts[3] = ?
      AND d.in_leandojo
      AND NOT d.name LIKE '%._private%'
      AND NOT d.name LIKE '%._simp_%'
      AND NOT d.name LIKE '%._proof_%'
      AND NOT d.name LIKE '%.match_%'
      AND (? OR ds.docstring IS NOT NULL)
    ORDER BY d.module, d.name
    LIMIT ?
""", [SUBAREA, not ONLY_WITH_DOCSTRING, LIMIT]).fetchall()

print(f"{len(rows)} declarations in Mathlib.LinearAlgebra.{SUBAREA}")

# ── render ────────────────────────────────────────────────────────────────
parts = ['<style>'
         '.decl-block { font-family: monospace; margin: 6px 0 14px 0; }'
         '.decl-name  { font-size: 1.05em; font-weight: bold; color: #1a1aff; }'
         '.decl-kind  { font-size: 0.8em; color: #888; margin-left: 6px; }'
         '.decl-mod   { font-size: 0.75em; color: #aaa; }'
         '.decl-doc   { margin: 3px 0 0 12px; color: #222; white-space: pre-wrap; }'
         '.decl-sig   { margin: 3px 0 0 12px; color: #555; font-size: 0.85em; white-space: pre-wrap; }'
         '.no-doc     { margin: 3px 0 0 12px; color: #bbb; font-style: italic; }'
         '.named      { background: #fffbe6; border-left: 3px solid #f0a000; padding-left: 6px; }'
         'hr.thin     { border: none; border-top: 1px solid #eee; margin: 2px 0; }'
         '</style>']

import re

def fmt_doc(doc):
    if not doc:
        return ''
    # bold **text** → <strong>
    doc = re.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', doc)
    # backtick → <code>
    doc = re.sub(r'`([^`]+)`', r'<code>\1</code>', doc)
    return doc

for name, kind, module, type_sig, doc in rows:
    is_named = bool(doc and re.search(r'\*\*', doc))
    named_cls = ' named' if is_named else ''
    parts.append(f'<div class="decl-block{named_cls}">')
    parts.append(f'  <span class="decl-name">{name}</span>'
                 f'  <span class="decl-kind">{kind or ""}</span>'
                 f'  <span class="decl-mod">{module or ""}</span>')
    if doc:
        parts.append(f'  <div class="decl-doc">{fmt_doc(doc)}</div>')
    elif type_sig:
        short_sig = (type_sig[:180] + '…') if len(type_sig) > 180 else type_sig
        parts.append(f'  <div class="decl-sig">{short_sig}</div>')
    else:
        parts.append(f'  <div class="no-doc">(no docstring)</div>')
    parts.append('</div><hr class="thin">')

display(HTML('\n'.join(parts)))

200 declarations in Mathlib.LinearAlgebra.Matrix


## Named theorems only (across all of LinearAlgebra)

In [4]:
rows = db.execute("""
    SELECT d.name, d.kind, d.module, ds.docstring
    FROM docstrings ds
    JOIN declarations d ON d.name = ds.name
    WHERE d.module LIKE 'Mathlib.LinearAlgebra%'
      AND regexp_matches(ds.docstring, '\\*\\*[^*]+\\*\\*')
    ORDER BY d.module, d.name
""").fetchall()

print(f"{len(rows)} declarations with named-theorem docstrings")

parts = ['<style>'
         '.nt-block { font-family: monospace; margin: 8px 0 16px 0;'
         '            border-left: 4px solid #f0a000; padding-left: 10px; background: #fffbe6; }'
         '.nt-name  { font-weight: bold; color: #1a1aff; font-size: 1.05em; }'
         '.nt-kind  { color: #888; font-size: 0.8em; margin-left: 6px; }'
         '.nt-mod   { color: #aaa; font-size: 0.75em; }'
         '.nt-doc   { margin-top: 4px; white-space: pre-wrap; }'
         '</style>']

for name, kind, module, doc in rows:
    doc_fmt = re.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', doc or '')
    doc_fmt = re.sub(r'`([^`]+)`', r'<code>\1</code>', doc_fmt)
    parts.append(
        f'<div class="nt-block">'
        f'<span class="nt-name">{name}</span>'
        f'<span class="nt-kind">{kind or ""}</span>'
        f'<span class="nt-mod"> — {module or ""}</span>'
        f'<div class="nt-doc">{doc_fmt}</div>'
        f'</div>'
    )

display(HTML('\n'.join(parts)))

30 declarations with named-theorem docstrings


## Declarations WITHOUT docstrings (what's missing)

Flip through these to get a sense of what isn't documented and why.

In [5]:
SUBAREA2 = 'Dimension'   # change this

rows = db.execute("""
    SELECT d.name, d.kind, d.type_sig
    FROM declarations d
    LEFT JOIN docstrings ds ON d.name = ds.name
    WHERE d.module_parts[1] = 'Mathlib'
      AND d.module_parts[2] = 'LinearAlgebra'
      AND d.module_parts[3] = ?
      AND d.in_leandojo
      AND ds.name IS NULL
      AND NOT d.name LIKE '%._private%'
      AND NOT d.name LIKE '%._simp_%'
      AND NOT d.name LIKE '%._proof_%'
      AND NOT d.name LIKE '%.match_%'
    ORDER BY d.name
    LIMIT 150
""", [SUBAREA2]).fetchall()

print(f"{len(rows)} declarations in {SUBAREA2} with no docstring (capped at 150)")

parts = ['<style>'
         '.nd-block { font-family: monospace; margin: 4px 0; }'
         '.nd-name  { color: #555; }'
         '.nd-kind  { color: #aaa; font-size: 0.8em; margin-left: 4px; }'
         '.nd-sig   { color: #999; font-size: 0.82em; margin-left: 12px; white-space: pre-wrap; }'
         '</style>']

for name, kind, type_sig in rows:
    short = (type_sig[:160] + '…') if type_sig and len(type_sig) > 160 else (type_sig or '')
    parts.append(
        f'<div class="nd-block">'
        f'<span class="nd-name">{name}</span>'
        f'<span class="nd-kind">{kind or ""}</span>'
        f'<div class="nd-sig">{short}</div>'
        f'</div>'
    )

display(HTML('\n'.join(parts)))

150 declarations in Dimension with no docstring (capped at 150)


## Full-text search across all docstrings

In [6]:
SEARCH = 'Matrix determinant lemma'   # change this

rows = db.execute("""
    SELECT d.name, d.kind, d.module, ds.docstring
    FROM docstrings ds
    JOIN declarations d ON d.name = ds.name
    WHERE d.module LIKE 'Mathlib.LinearAlgebra%'
      AND lower(ds.docstring) LIKE lower(?)
    ORDER BY d.module, d.name
    LIMIT 60
""", [f'%{SEARCH}%']).fetchall()

print(f"{len(rows)} results for '{SEARCH}'")

for name, kind, module, doc in rows:
    # highlight the search term
    hi = re.sub(f'(?i)({re.escape(SEARCH)})', r'<mark>\1</mark>', doc or '')
    hi = re.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', hi)
    hi = re.sub(r'`([^`]+)`', r'<code>\1</code>', hi)
    display(HTML(
        f'<div style="font-family:monospace;margin:6px 0 12px 0">'
        f'<b style="color:#1a1aff">{name}</b> '
        f'<span style="color:#888;font-size:.8em">{kind or ""} — {module or ""}</span>'
        f'<div style="margin:3px 0 0 12px;white-space:pre-wrap">{hi}</div>'
        f'</div><hr style="border:none;border-top:1px solid #eee">'
    ))

3 results for 'Matrix determinant lemma'
